# EDA: S&P 500 Equity Volatility
Exploratory analysis of the OHLCV data and feature panel.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys; sys.path.insert(0,"..")
from src.config import FEATURES_PATH, OHLCV_DB_PATH
from src.data.storage import read_ohlcv
%matplotlib inline

## 1. Universe & Coverage

In [ ]:
panel = pd.read_parquet(FEATURES_PATH)
print(f"Rows: {len(panel):,}  Tickers: {panel['ticker'].nunique()}  Dates: {panel['date'].min().date()} to {panel['date'].max().date()}")
panel.head()

## 2. Return Distribution

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(12,4))
panel["ret_lag_1"].hist(bins=200, ax=axes[0]); axes[0].set_title("Daily log returns")
panel["log_rv_5_next"].hist(bins=200, ax=axes[1]); axes[1].set_title("Target: log RV-5 (next 5d)")
plt.tight_layout()

## 3. Volatility Clustering

In [ ]:
spx = panel[panel["ticker"]=="AAPL"].sort_values("date")
fig, ax = plt.subplots(figsize=(14,4))
ax.plot(spx["date"], spx["rv_21d"], linewidth=0.8)
ax.set_title("AAPL: 21-day trailing realized vol")
ax.axvspan("2020-02-01","2020-06-01", alpha=0.2, color="red", label="COVID")
ax.legend(); plt.tight_layout()

## 4. Cross-sectional Feature Correlations

In [ ]:
from src.config import TARGET_COL
feat_cols = [c for c in panel.columns if c not in ("ticker","date",TARGET_COL)]
corr = panel[feat_cols + [TARGET_COL]].corr()[TARGET_COL].drop(TARGET_COL).sort_values()
fig, ax = plt.subplots(figsize=(8,10))
corr.plot.barh(ax=ax); ax.set_title("Feature correlation with target"); plt.tight_layout()